# Your prepared video — GPU render
1. Select **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU).
2. Press **Run all**. Everything is included; there is no file upload or runtime GitHub token to enter.
The video previews and downloads when finished. On error, an error-report ZIP downloads instead. First-run tool downloads take additional time. GPU availability is controlled by Colab.
Current video: **The Last Cookie**, an original 7.4-second production-rig test. Live CUDA performance has not yet been measured.
Character: Snow Rig © Blender Foundation | studio.blender.org — CC BY 4.0.

In [ ]:
#@title Render the prepared video on the Colab GPU
from pathlib import Path
import base64, collections, hashlib, io, json, subprocess, sys, time, zipfile
from IPython.display import Video, display
from google.colab import files
PAYLOAD = '...'
EXPECTED_SHA256 = '6d156a51784ed983fedf92ff65f22799c553e50cd90b8b26b3aa968341f5327b'
data = base64.b64decode(PAYLOAD)
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256:
    raise ValueError("The embedded package is damaged. Reopen the latest notebook.")
destination = Path('/content').resolve()
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    if any(not (destination/n).resolve().is_relative_to(destination) for n in archive.namelist()):
        raise ValueError("Unsafe package member")
    archive.extractall(destination)
del PAYLOAD, data
root = Path('/content/video_engine')
log = root/'renderer.log'
tail = collections.deque(maxlen=70)
started = time.monotonic()
print("Prepared scene loaded. Checking GPU, then installing the verified renderer and character.", flush=True)
with log.open('w', encoding='utf-8') as logfile:
    proc = subprocess.Popen([sys.executable, '-u', str(root/'run_colab_render.py'), '--root', str(root)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        logfile.write(line)
        tail.append(line)
        if line.startswith(('GPU_STAGE', 'BUILD_STAGE', 'FRAME_PROGRESS', 'GPU_VIDEO_READY', 'GPU_ERROR')):
            print(line.rstrip(), flush=True)
    code = proc.wait()
if code:
    print(''.join(tail))
    report = root/'gpu-error-report.zip'
    with zipfile.ZipFile(report, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in [log, root/'GPU_ERROR.txt', root/'gpu-status.json']:
            if path.exists(): archive.write(path, path.name)
    print("Render failed. Send the downloaded error report back to the chat.")
    files.download(str(report))
    raise RuntimeError(f"GPU render failed with exit code {code}. The complete log is in the error report.")
result = json.loads((root/'gpu-status.json').read_text())
output = Path(result['output'])
print(json.dumps({'status':'complete', 'total_seconds':round(time.monotonic()-started, 1),
                  'renderer':result['renderer'], 'render_and_encode_seconds':result['render_and_encode_seconds']}, indent=2))
display(Video(str(output), embed=True))
files.download(str(output))
print("Done. Send the video or the output above back to the chat.")
